# نقل ملفات الحلقة إلى جلسة Colab

هذه الدفترية عامة وموجودة في مستودع GitHub عام؛ يمكن فتحها في Colab دون إعداد GitHub Token أو OAuth Client. **لا تحتوي على أي بيانات اعتماد.**

للوصول إلى ملفات Drive الخاصة، ستطلب خلية `drive.mount` موافقة حساب Google بالطريقة المدمجة في Colab. هذا ليس إعداد OAuth يدوياً، ولا يمكن تجاوزه بأمان لملفات خاصة.

تنسخ الدفترية ملفات المصدر والدبلجة وتقارير التوقيت والماستر الصوتي إلى مساحة الجلسة، وتكتب تقرير تحقق في مجلد `مخرجات المزامنة`. لا تبدأ الرندر أو تغير أي ملف أصلي.

In [ ]:
# 1) إعداد الأدوات الأساسية وفحص المساحة
!apt-get -qq update && apt-get -qq install -y ffmpeg
import os, shutil, subprocess, json, hashlib, zipfile, time
from pathlib import Path
print(subprocess.run(['ffmpeg','-version'],capture_output=True,text=True).stdout.splitlines()[0])
print('Free disk GiB:', round(shutil.disk_usage('/content').free/1024**3,2))


In [ ]:
# 2) ربط Google Drive بالطريقة المدمجة في Colab (موافقة الحساب تظهر عند الحاجة)
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted:', Path('/content/drive/MyDrive').exists())


In [ ]:
# 3) تحديد مجلد الحلقة ومجلد المخرجات
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive')
EPISODE_DIR = DRIVE_ROOT / 'دبلجة الحلقة الجديدة'
OUTPUT_DIR = EPISODE_DIR / 'مخرجات المزامنة'
WORK_DIR = Path('/content/episode_inputs')
WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not EPISODE_DIR.exists():
    raise FileNotFoundError(f'لم أجد مجلد الحلقة: {EPISODE_DIR}. تحقق من أن الحساب المختار هو حساب Drive الذي يحتوي الحلقة.')
print('Episode folder:', EPISODE_DIR)
print('Output folder:', OUTPUT_DIR)


In [ ]:
# 4) نسخ ملفات الإدخال الموجودة فقط، دون تعديل الأصول
EXPECTED = [
    '🔴 FULL - 女孩相亲认错人，将错就错与千亿霸总当天领证！ - Original Quality.mkv',
    'الحلقة_مدبلجة_بالعربية.mkv',
    'تقارير_التحقق_والموائمة.zip',
    'الأصوات_العربية_المتبقية_06-10.zip',
    '01_لي_يانغ_Leda.wav',
    '02_لو_مينغ_Achird.wav',
    '03_تشوانغ_شيان_يوي_Kore.wav',
    '04_فو_شاوتشينغ_Puck.wav',
    '05_شيا_مو_يان_Zephyr.wav',
]
manifest = {'copied': [], 'missing': [], 'errors': [], 'created_at': time.strftime('%Y-%m-%dT%H:%M:%S%z')}
for name in EXPECTED:
    src = EPISODE_DIR / name
    if not src.is_file():
        manifest['missing'].append(name)
        continue
    dst = WORK_DIR / name
    try:
        if not dst.exists() or dst.stat().st_size != src.stat().st_size:
            shutil.copy2(src, dst)
        manifest['copied'].append({'name': name, 'size_bytes': dst.stat().st_size})
        print('COPIED', name, dst.stat().st_size)
    except Exception as e:
        manifest['errors'].append({'name': name, 'error': repr(e)})
        print('ERROR', name, repr(e))
print('Copied:', len(manifest['copied']), 'Missing:', len(manifest['missing']), 'Errors:', len(manifest['errors']))


In [ ]:
# 5) فحص سلامة ZIPs ومدد الفيديوهات وحساب SHA-256
def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''):
            h.update(b)
    return h.hexdigest()
for name in ['تقارير_التحقق_والموائمة.zip','الأصوات_العربية_المتبقية_06-10.zip']:
    p=WORK_DIR/name
    if p.exists():
        try:
            with zipfile.ZipFile(p) as z:
                bad=z.testzip(); manifest.setdefault('zip_checks',{})[name]={'ok':bad is None,'bad_member':bad,'members':z.namelist()[:80]}
        except Exception as e:
            manifest.setdefault('zip_checks',{})[name]={'ok':False,'error':repr(e)}
for name in EXPECTED[:2]:
    p=WORK_DIR/name
    if p.exists():
        probe=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration,size','-of','json',str(p)],capture_output=True,text=True)
        if probe.returncode==0:
            info=json.loads(probe.stdout).get('format',{})
            manifest.setdefault('media',{})[name]={'duration_seconds':float(info.get('duration',0)),'size_bytes':int(info.get('size',p.stat().st_size)),'sha256':sha256_file(p)}
        else:
            manifest.setdefault('media',{})[name]={'ffprobe_error':probe.stderr[-1500:]}
manifest['status']='inputs_ready' if not manifest['missing'] and not manifest['errors'] else 'incomplete'
print(json.dumps(manifest,ensure_ascii=False,indent=2))


In [ ]:
# 6) حفظ تقرير النقل والتحقق في مجلد مخرجات المزامنة على Drive
report_path = OUTPUT_DIR / 'colab_transfer_manifest.json'
report_path.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('Saved:', report_path)
print('Status:', manifest['status'])
print('Local inputs:', WORK_DIR)
print('هذه المرحلة تنقل وتتحقق فقط؛ لا تبدأ مزامنة الشفاه أو الرندر تلقائياً.')
